# Embedding
![rag_embedding](figures/rag_embedding.png)

- 분할된 텍스트를 벡터 표현(임베딩 벡터)으로 변환한다.
- 기존 언어모델은 토큰을 기준으로 임베딩 + 내부구조에서 해결하지만 별도의 임베딩 모델은 문장을 기준으로 이를 출력해준다.
- LangChain은 OpenAI, HuggingFace 등 다양한 임베딩 모델을 지원하며, 동일한 인터페이스로 사용할 수 있다.
- [임베딩모델의 메서드](https://api.python.langchain.com/en/latest/embeddings/langchain_core.embeddings.embeddings.Embeddings.html#langchain_core.embeddings.embeddings.Embeddings)

    - **`embed_documents(texts: List[str])`**
        - 여러 문서를 받아 벡터화(임베딩)한다.
        - Context를 벡터화 할 때 사용한다.
    - **`embed_query(text: str)`**
        - 하나의 문자열(문서)을 받아 벡터화한다.
        - Query를 벡터화 할 때 사용한다.


In [1]:
docs = [
        "나는 고양이와 개 중 반려동물로 개를 키우고 싶습니다.",
        "이 강아지 품종은 진도개 입니다. 국제 표준으로 중대형견으로 분류되며 다리가 길어 체고가 높은 편에 속합니다.",
        "日本の市内バスの運賃は主に距離によって決まり、地域やバス会社によって異なる場合があります",                  # 일본의 시내버스 요금은 주로 거리에 따라 결정되며, 지역 및 버스 회사에 따라 다를 수 있습니다.
        "Bus fares in the United States vary from city to city, but are generally around $2.90 for a regular bus.",  # 미국의 버스 요금은 도시마다 다르지만, 일반적으로 정기 버스의 경우 2.90달러 정도입니다.
        "광역버스 요금은 일반 3000원, 청소는 1800원, 어린이 1500원 입니다.", 
]

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [5]:
#############################
#  OpenAI의 Embedding 모델.
from langchain_openai import OpenAIEmbeddings#, ChatOpenAI

# e_model_id = "text-embedding-3-small" # "text-embedding-3-large"
e_model_id = "text-embedding-3-large"
embedding_model = OpenAIEmbeddings(model=e_model_id)

In [39]:
################################
# Ollama Embedding 모델
from langchain_ollama import OllamaEmbeddings

e_model_id = "bge-m3"
embedding_model = OllamaEmbeddings(model=e_model_id)

In [2]:
##############################
# Huggingface Embedding Model
#  Hugging-hub:  Model > task - NLP > Feature Extraction 
from langchain_huggingface import HuggingFaceEmbeddings

e_model_id = "intfloat/multilingual-e5-large"
embedding_model = HuggingFaceEmbeddings(model=e_model_id)

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

c:\Users\jinhy\anaconda3\envs\lang_env\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jinhy\.cache\huggingface\hub\models--intfloat--multilingual-e5-large. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


README.md:   0%|          | 0.00/160k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

In [6]:
docs

['나는 고양이와 개 중 반려동물로 개를 키우고 싶습니다.',
 '이 강아지 품종은 진도개 입니다. 국제 표준으로 중대형견으로 분류되며 다리가 길어 체고가 높은 편에 속합니다.',
 '日本の市内バスの運賃は主に距離によって決まり、地域やバス会社によって異なる場合があります',
 'Bus fares in the United States vary from city to city, but are generally around $2.90 for a regular bus.',
 '광역버스 요금은 일반 3000원, 청소는 1800원, 어린이 1500원 입니다.']

In [7]:
# 문서들을 embedding
embedded_docs = embedding_model.embed_documents(docs)

In [8]:
type(embedded_docs), type(embedded_docs[0])

(list, list)

In [9]:
import numpy as np

np.shape(embedded_docs)
# (5: 문서개수, 1536: 개별 문서의 Embedding Vector 차원)

(5, 1024)

In [10]:
embedded_docs[0]
docs[0]

'나는 고양이와 개 중 반려동물로 개를 키우고 싶습니다.'

In [3]:
import numpy as np

def cosine_similarity(v1:np.ndarray|list, v2:np.ndarray|list) -> float:
    # v1과 v2의 코사인 유사도를 계산.
    # -1 ~ 1
    # 1: 같은것.  0: 관계없는 것, -1: 반대
    v1 = np.array(v1)
    v2 = np.array(v2)
    return (v1 @ v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

In [12]:
#### 질문 -> Embedding Vector로 변환 
#         -> 문서들의 Embedding Vector들과 유사도를 계산
#         -> 유사도 높은 순서로 N개의 문서를 반환.

# query = "당신이 좋아하는 동물은 무엇인가요?"
# query = "진도개는 한국산 개 품종입니까?"
query = "성인 버스 요금은 얼마인가요?"
embedded_query = embedding_model.embed_query(query)
np.shape(embedded_query), type(embedded_query)

((1024,), list)

In [13]:
# embedded_query와 embedded_docs 간의 유사도 계산.
for i, ev in enumerate(embedded_docs):
    print(f"{i+1}. {cosine_similarity(ev, embedded_query)}")

1. 0.7468271368563301
2. 0.7260714506668857
3. 0.7683728973276016
4. 0.791565059418625
5. 0.8513893151689992


# 벡터 데이터베이스(Vector Database)
- Embedding 된 문서를 Vector Database(Vector Store)에 저장한다.
- 이후 질문(Query)와 관련된 내용을 유사도를 이용해 검색해 질문과 함께 prompt로 만든다. (Retrieve)

![rag_vector_store](figures/rag_vector_store.png)

## 벡터 데이터베이스란
- 벡터 임베딩을 저장하고 관리하는 데이터베이스를 의미한다.
- 모든 데이터는 적절한 임베딩 모델을 활용하면 임베딩 벡터로 변환할 수 있다. 이렇게 변환된 임베딩 벡터를 벡터 데이터베이스에 저장하면 **임베딩 벡터 간의 거리 계산을 통해 데이터 간 유사도를 검색할 수 있다.**
    - **이미지, 텍스트, 음성 등 비정형 데이터**를 임베딩 모델로 **벡터화한 뒤 데이터베이스에 저장**한다.
    - 벡터 간의 **유사도 계산**을 통해 연관성 있는 데이터나 유사한 데이터를 효과적으로 검색할 수 있다.
    - 좋은 검색 결과를 위해서는 벡터의 품질이 중요하다. 그래서 **임베딩 모델(Embedding Model)을 잘 선택하는 것이 중요**하다.
- 벡터 데이터베이스는 이러한 벡터 간 거리 계산에 특화된 데이터베이스다.

## 주요 특징

- **고차원 벡터 저장**
  -  벡터 데이터베이스는 수백에서 수천 차원에 이르는 벡터 데이터를 효율적으로 저장하고 관리한다. 
  -  전통적인 데이터베이스로는 어려운 고차원 벡터 간 유사도 검색을 효율적으로 수행한다.
- **유사성 기반 검색**
  -  벡터 간의 거리를 측정하여 유사한 데이터를 빠르게 검색할 수 있다. 
  -  일반적으로 사용되는 거리계산기법은 다음과 같다.
     - 코사인 유사도(Cosine Similarity)
     - 유클리드 거리(Euclidean Distance)
         - 보통 이미지. 이미지의 경우 벡터의 크기도 중요한 경우가 많음
     - 맨하탄 거리(Manhattan Distance) 
- 비정형 데이터 처리: 텍스트, 이미지, 오디오 등 다양한 비정형 데이터를 벡터로 변환하여 저장하고, 이러한 데이터를 효과적으로 검색할 수 있다.

## 벡터 데이터베이스와 딥러닝
- 벡터 데이터베이스는 딥러닝 기술의 발전과 깊은 관련이 있다.
- 딥러닝 모델은 학습 과정에서 데이터의 특징을 추출하는 방법을 함께 학습한다. 충분한 데이터를 학습한 딥러닝 모델은 **데이터의 특성을 설명하는 특성 벡터(feature vector)를 효과적으로 생성**할 수 있다.
- 이때 추출된 특성 벡터는 고차원 데이터(RAW Data)를 저차원 공간에서 표현한 **임베딩 벡터**다.
    - > **임베딩**은 고차원 데이터를 저차원 공간으로 변환하여 표현하는 방법으로, 정보 손실을 최소화하면서 데이터 간의 의미 있는 관계를 벡터 공간에서 유지한다.
- 딥러닝 모델로 추출한 데이터의 특징(feature vector)을 임베딩 공간에 배치하면, 비슷한 데이터는 가까이, 그렇지 않은 데이터는 멀리 배치된다.
- 이러한 특성을 활용하면 임베딩 벡터 간의 거리를 계산해 유사한 데이터를 효과적으로 검색할 수 있다. 벡터 데이터베이스는 이러한 임베딩 벡터의 특성을 기반으로 개발되었다.
- 딥러닝 기술의 발전과 폭넓은 활용으로 임베딩 데이터의 사용이 증가하면서, 이를 저장하고 관리하는 기능에 특화된 데이터베이스에 대한 수요도 증가해 다양한 벡터 데이터베이스가 등장했다.

## 벡터 데이터베이스의 주요 기능
1. **저장**  
   - 이미지, 텍스트, 음성 등 **비정형 데이터**를 임베딩 모델을 통해 벡터로 변환한 뒤 벡터 데이터베이스에 저장한다.
2. **검색**  
   - 검색하려는 데이터를 임베딩 모델로 변환한 뒤, 벡터 데이터베이스에서 **유사도를 기반**으로 검색한다.
3. **결과 반환**  
   - 벡터 데이터베이스는 저장된 벡터 중 검색 쿼리 임베딩과 가장 가까운 벡터를 찾아 반환한다.

## LLM과 벡터 데이터베이스
- ChatGPT(LLM)의 등장 이후 벡터 데이터베이스는 폭발적인 주목을 받았다.
- 임베딩 벡터의 유사도를 기반으로 문서를 검색하는 RAG(Relevant Augmented Generation) 기술은 LLM의 환각(할루시네이션) 현상을 줄이고, LLM을 추가 학습하지 않고도 최신 정보를 효율적으로 활용할 수 있는 핵심 기법으로 자리 잡았다.
   


## 벡터 데이터베이스 종류
![img](figures/vector_database.png)

<<https://blog.det.life/why-you-shouldnt-invest-in-vector-databases-c0cd3f59d23c>>

### 주요 벡터 데이터베이스 종류
- **Pinecone**
    - 클라우드 기반의 완전 관리형 벡터 데이터베이스 서비스로, 간단한 API를 통해 벡터 데이터를 관리할 수 있다.  
    - 자동 확장성과 고가용성을 제공하며, 실시간 데이터 수집과 유사성 검색에 최적화되어 있다.
    - 가장 쉽게 시작할 수 있는 관리형 서비스를 제공한다.
- **Chroma**
    - 벡터 임베딩을 효율적으로 저장하고 검색할 수 있는 오픈소스 데이터베이스로, AI 및 머신러닝 애플리케이션에 최적화되어 있다.
    - 대규모 임베딩 저장에 최적화되어 있다.
- **FAISS**
    - Facebook AI에서 개발한 고성능 벡터 검색 라이브러리로, 고차원 벡터의 효율적인 유사성 검색을 위해 최적화되어 있다.
    - GPU를 활용해 계산 성능을 높이며, 벡터 양자화 기술을 활용하여 메모리 사용을 최적화한다.
    - 근사 최근접 이웃 검색(ANNS)에 최적화되어 있다.
- **Milvus**
    - 오픈소스 벡터 데이터베이스로, 대규모 벡터 데이터를 효율적으로 저장하고 검색할 수 있다.  
    - 분산 아키텍처를 채택하여 확장성이 뛰어나며, IVF_PQ, DiskANN 등 다양한 인덱싱 알고리즘을 지원한다.
    - 대규모 데이터셋 처리에 가장 적합한 솔루션이다.
- **Weaviate**
    - 오픈소스 벡터 데이터베이스로, 텍스트, 이미지, 오디오 등 다양한 비정형 데이터를 벡터로 저장하고 검색할 수 있다.  
    - GraphQL API를 통해 접근 가능하며, 내장된 머신러닝 모듈을 통해 가장 강력한 의미론적 검색 기능을 제공한다.
- **Qdrant**
    - Rust로 개발된 고성능 벡터 검색 엔진으로, 실시간 근사 최근접 이웃 검색을 제공한다.  
    - 추천 시스템에 특화되어 있으며, 벡터 임베딩 저장과 유사도 쿼리를 효율적으로 수행한다.
- **Elasticsearch**
    - HNSW 알고리즘을 사용하여 벡터 검색을 구현하는 검색 엔진이다.
    - 전통적인 검색 기능과 벡터 검색을 효과적으로 결합할 수 있어, 하이브리드 검색에 가장 적합하다.
- **PGVector**
    - PostgreSQL의 확장 모듈로, 벡터 데이터를 저장하고 유사성 검색을 수행할 수 있게 해준다.  
    - SQL과 통합된 벡터 연산이 가능하며, L2 거리, 코사인 거리, 내적 등 다양한 거리 측정 방식을 지원한다.


# Langchain - Vector Store 연동 
- Langchain은 다양한 벡터 데이터베이스와 연동할 수 있다.
- 벡터 데이터베이스 마다 API가 다르기 때문에, Langchain을 사용하면 동일한 interface로 사용할 수 있다.

## **VectorStore**
- Langchain이 지원하는 모든 벡터 데이터베이스는 **VectorStore** 인터페이스를 구현한다.
- 그래서 Langchain에서는 벡터 데이터베이스를 **Vector Store** 라고 한다.
- https://python.langchain.com/docs/integrations/vectorstores/

### Vector Store 연결
- Vector DB와 연결하는 메소드
- `VectorStore.from_document()`
  - Document들을 insert 하면서 연결.
  - Database가 있으면 연결, 없으면 생성하면서 연결한다.
  - Parameter
    - documents: insert할 문서들을 list[Document]로 전달.
    - embedding model
    - vector db에 연결하기 위한 설정들을 넣어준다.
-`VectorStore()`
  - vector db와 연결만 한다.
  - Database가 있으면 연결, 없으면 생성하면서 연결한다.
  - Parameter
    - embedding model
    - vector db에 연결하기 위한 설정들을 넣어준다.
## InMemoryVectorStore
- langchain에서 제공하는 메모리 기반 벡터 데이터베이스이다.
- Data들을 Dictionary를 사용해 메모리에 저장하며, 검색 할 때 코사인 유사도(cosine similarity)를 계산하여 조회한다.

In [15]:
from dotenv import load_dotenv

load_dotenv()

True

In [16]:
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")
# VectorStore 생성시 Embedding 모델을 넣어 생성한다.
## DB 연결
vector_store = InMemoryVectorStore(
    embedding=embedding_model
)

In [ ]:
vector_store

In [18]:
# 문서 정의
from langchain_core.documents import Document
d1 = Document(id="1",  # 문서 ID(식별자)
            page_content="Apple, Pear, Watermelon", # 문서 내용
            metadata={"category":"fruit"}, # 문서 정보
            )
d2 = Document(id="2", page_content="Python, C++, Java, C#, Rust", 
            metadata={"category":"IT"})
d3 = Document(id="3", page_content="Football, Baseball, Basketball", 
            metadata={"category":"sports"})

docs = [d1, d2, d3]
# VectorDB에 저장
vector_store.add_documents(documents=docs)

['1', '2', '3']

In [ ]:
# DB와 연결하면서 document들을 insert/upsert
vector_store2 = InMemoryVectorStore.from_documents(
    documents=docs,
    embedding=embedding_model
)

In [19]:
# 검색 - Query와 유사한 문서를 Vector Store에서 찾기.
query = "SQL"
query = "야구"
query = "오렌지"
# result = vector_store.similarity_search(
result = vector_store.similarity_search_with_score( # 검색한 결과 + 유사도 점수
    query=query, # 찾을 질문
    k=2,         # 몇개 문서를 찾을지 지정.
)

In [20]:
result

[(Document(id='1', metadata={'category': 'fruit'}, page_content='Apple, Pear, Watermelon'),
  0.16768911339041154),
 (Document(id='2', metadata={'category': 'IT'}, page_content='Python, C++, Java, C#, Rust'),
  0.11882075375938522)]

In [1]:
# self-shot
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv
load_dotenv()
# 벡터 스토어 생성
embedding_model = OpenAIEmbeddings(model='text-embedding-3-large')
texts = ["apple", "banana", "sports"]
vector_store = InMemoryVectorStore.from_texts(
	texts=texts,
    embedding=embedding_model
)
# 질문 -> 임베딩 벡터 변환
query = "너가 좋아하는 과일이 뭐야?"
embed_query = embedding_model.embed_query(query)
# 유사도 검사

In [ ]:
# 유사도 점수들 계산
vector_store.max_marginal_relevance_search(query)
# vector_store.max_marginal_relevance_search_by_vector(embed_query)
# vector_store.similarity_search(query)
vector_store.similarity_search_with_score(query, k=3)

[(Document(id='534800f1-94ee-4b45-8eaf-0c97035167e6', metadata={}, page_content='banana'),
  0.3235796007353364),
 (Document(id='5c6e63ff-484b-482b-9040-751be0bc2230', metadata={}, page_content='apple'),
  0.28703597738283226),
 (Document(id='f664df6e-db49-4d93-b501-2f5c0ffd960a', metadata={}, page_content='sports'),
  0.0875708293452917)]

In [ ]:
len(vector_store.store)
vector_store.store
vector_store.store['5c6e63ff-484b-482b-9040-751be0bc2230'].keys()

{'5c6e63ff-484b-482b-9040-751be0bc2230': {'id': '5c6e63ff-484b-482b-9040-751be0bc2230',
  'vector': [-0.020793559029698372,
   0.014009363017976284,
   -0.0008607447962276638,
   0.01870741881430149,
   -0.008157994598150253,
   -0.0020034576300531626,
   0.0038118697702884674,
   0.021624622866511345,
   0.016061581671237946,
   0.03186875581741333,
   0.017689788714051247,
   -0.022489607334136963,
   0.014077205210924149,
   0.01721489615738392,
   -0.0173420999199152,
   0.022336963564157486,
   0.02237088419497013,
   -0.012313314713537693,
   -0.020759638398885727,
   -0.009735320694744587,
   0.0160276610404253,
   -0.053357698023319244,
   0.013534469529986382,
   0.02859538421034813,
   0.016782402992248535,
   0.0048125386238098145,
   -0.003625304438173771,
   -0.0004327574570197612,
   -0.00375462812371552,
   0.02571210078895092,
   0.03548134118318558,
   0.031037693843245506,
   -0.009141703136265278,
   0.005596961360424757,
   -0.055121585726737976,
   0.03677034005522

<!-- # 실습
- `data/olympic.txt`

1. text loading
2. text split
3. embedding + vector store(InMemoryVectorStore)에 저장
4. query(질의) -->

In [33]:
from dotenv import load_dotenv
load_dotenv()

True

In [1]:
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
# 1. 문서 loading + 2. Split(Chunking)
loader = TextLoader("data/olympic.txt", encoding="utf-8")
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, chunk_overlap=50
)
docs = loader.load_and_split(splitter)
# raw_docs = loader.load() -> docs = splitter.split_documents(raw_docs)
print(len(docs))

61


In [5]:
docs[0]

Document(metadata={'source': 'data/olympic.txt'}, page_content='올림픽')

In [4]:
# 3. embedding + 4. VectorStore에 저장
# VectorDB와 연결
vector_store = InMemoryVectorStore(
    embedding=OpenAIEmbeddings(model="text-embedding-3-small")  #임베딩모델
)
# 저장(embedding 처리후 저장 -> vector store가 임베딩 작업은 처리한다.)
vector_store.add_documents(docs)

['7dfab89d-0a22-4faf-9fd3-d6938e2c4e44',
 'c3f98b52-2ea6-45fa-849e-2e88af738f11',
 '6e3ed87c-87b9-4e93-931e-bb5fefcd9e4b',
 'efb3d5f0-bbf9-4db5-ae70-7f94496165a7',
 'a4bde663-5b91-420a-804f-c5d37a395e00',
 '03304f52-340a-486a-9ac0-1a6117f6574b',
 'de619d94-aea0-4cb6-b411-54c57de8d632',
 '35dcb905-758d-4e67-ba58-7e9867ba20af',
 '691a9a38-4551-4366-8e7f-578303100c92',
 '2d36446d-d7be-4368-be0d-fc87ec7c1845',
 '69d1bb06-88f3-4d03-8363-234904f562f1',
 '22800907-4b50-4d94-abed-463ec42ec44a',
 'b8ce0a6d-09f8-47b4-8c09-0071c6fd8470',
 'c22b8b9e-b9b1-418c-be59-e436a5079fd8',
 '78ebd153-9e66-4bc6-bb08-5191b923c8ec',
 'df380ca2-975a-4d2c-9c08-d01eea46e534',
 '06e5558e-f488-4de6-b103-28429a9fa354',
 'b30f0923-7667-4db9-957c-929098d808f1',
 '931e2c19-cdc4-4674-943a-61f3fbcb2f1f',
 '6ea36543-e268-411e-a8f7-c87de8897003',
 '68337c68-1fe3-4b62-86a7-62250252e3f8',
 '7d5852b7-f4bd-46ab-b0d4-5831a42270b8',
 '45272984-96c0-483c-b496-be6e312c50eb',
 '01f2a649-4126-4dd9-a89b-4f63b7acadb2',
 '8f135267-f9c0-

In [ ]:
# 연결 + 저장
# v2 = InMemoryVectorStore.from_documents(
#     embedding=OpenAIEmbeddings("text-embedding-3-small"),
#     documents=docs
# )

In [9]:
# 질문 -> 의미적 유사도가 높은 k(5)개의 문서를 반환.
query = "동계 올림픽에 대해 설명해주세요."    #input("질문:")
results = vector_store.similarity_search_with_score(
    query=query, 
    k=5
)
for result in results:
    print(result[1], result[0].page_content[:200])

0.5847594806136202 하계올림픽
0.5177327762755929 동계올림픽
동계 올림픽은 눈과 얼음을 이용하는 스포츠들을 모아 이루어졌으며 하계 올림픽 때 실행하기 불가능한 종목들로 구성되어 있다. 피겨스케이팅, 아이스하키는 각각 1908년과 1920년에 하계올림픽 종목으로 들어가 있었다. IOC는 다른 동계 스포츠로 구성된 새로운 대회를 만들고 싶어 했고, 로잔에서 열린 1921년 올림픽 의회에서 겨울판 올림픽을 열기
0.4875855187226654 올림픽
0.4748001481635575 오늘날의 올림픽
1896년 대회때는 14개국에서 241명의 선수단이 참가했지만 2008년 하계 올림픽때는 204개국에서 10,500명의 선수가 참가하는 등 세계적인 대회로 변모했다. 동계 올림픽의 규모는 하계 올림픽 규모보다 작다. 예를 들면 2006 토리노 동계 대회때는 80개국에서 2,508명의 선수가 참가했으며 82개 세부종목이 있었고, 2008 베이
0.4493941587211385 고대올림픽


## MMR(최대 한계 관련성-Maximal Marginal Relevance) 알고리즘 적용
최대 한계 관련성(Maximal Marginal Relevance, MMR) 알고리즘은 정보 검색 및 요약에서 검색 결과의 **관련성**과 **다양성**을 동시에 고려하여 최적의 결과를 제공하는 방법이다. 
이 알고리즘은 사용자 쿼리와의 관련성을 최대화하면서도 중복 정보를 최소화하여 다양한 정보를 제공하는 것을 목표로 한다.

1. **관련성과 다양성의 균형 조절**: MMR은 사용자 쿼리와 문서 간의 유사성 점수와 이미 선택된 문서들과의 다양성 점수를 조합하여 각 문서의 최종 점수를 계산한다. 이를 통해 관련성이 높으면서도 중복되지 않는 문서를 선택한다.

2. **수학적 정의**
   $$
   \text{MMR} = \lambda \cdot \text{Sim}(d, Q) - (1 - \lambda) \cdot \max_{d' \in D'} \text{Sim}(d, d')
   $$

   - $\text{Sim}(d, Q)$: 문서 $d$와 쿼리 $\text{Q}$ 사이의 유사성. (문서 유사성 계산)
   - $\max_{d' \in D'} \text{Sim}(d, d')$: 문서 $d$와 이미 선택된 문서 집합 $D'$ 중 가장 유사한 문서와의 유사성. (문서 다양성 계산)
   - $\lambda$: 유사성과 다양성의 중요도를 조절하는 매개변수(parameter)
3. **적용 분야**: MMR은 정보 검색, 추천 시스템, 문서 요약 등에서 활용된다. 특히 LLM 검색에서 성능 향상이 입증되었다.

### `vectorStore.max_marginal_relevance_search()` 메소드
  - MMR 알고리즘을 적용한 검색을 수행한다.
  - **파라미터**
    - **query**: 사용자로부터 입력받은 검색 쿼리
    - **k**: 최종적으로 선택할 문서의 수
    - **fetch\_k**: MMR 알고리즘 적용 시 고려할 상위 문서의 수
    - **lambda_mult**: 쿼리와의 유사성과 선택된 문서 간의 다양성 사이의 균형을 조절하는 매개변수. $\lambda = 1$이면 유사성만 고려하고, $\lambda = 0$이면 다양성만을 최대화한다.
    - **filter**: 검색 결과를 필터링할 조건을 지정한다.


In [ ]:
query = "동계 올림픽에 대해 설명해줘."
mmr_result = vector_store.max_marginal_relevance_search(
    query=query,
    k=5, #  최종 결과 문서 개수
    fetch_k=20, # 처음 검색할 문서개수.
    lambda_mult=0.5   # 1에 가까울수록 유사성에 0에 가까울 수록 다양성을 최대화.
)

for result in mmr_result:
    print(result.page_content[:200])
    print("------------------")

하계올림픽
------------------
동계올림픽
동계 올림픽은 눈과 얼음을 이용하는 스포츠들을 모아 이루어졌으며 하계 올림픽 때 실행하기 불가능한 종목들로 구성되어 있다. 피겨스케이팅, 아이스하키는 각각 1908년과 1920년에 하계올림픽 종목으로 들어가 있었다. IOC는 다른 동계 스포츠로 구성된 새로운 대회를 만들고 싶어 했고, 로잔에서 열린 1921년 올림픽 의회에서 겨울판 올림픽을 열기
------------------
올림픽
------------------
오늘날의 올림픽
1896년 대회때는 14개국에서 241명의 선수단이 참가했지만 2008년 하계 올림픽때는 204개국에서 10,500명의 선수가 참가하는 등 세계적인 대회로 변모했다. 동계 올림픽의 규모는 하계 올림픽 규모보다 작다. 예를 들면 2006 토리노 동계 대회때는 80개국에서 2,508명의 선수가 참가했으며 82개 세부종목이 있었고, 2008 베이
------------------
고대올림픽
------------------
